# 🏥 Robot Nurse Voice Control System
### Voice Cloning Edition — speaks in YOUR voice

**Stack:** Faster-Whisper (STT) → Llama 3.2 via Ollama (Intent) → Chatterbox (TTS, your voice clone) → Flask map + Gradio UI

> **Before running:** place your voice recording as `my_voice.m4a` (or `my_voice.wav`) in the same folder as this notebook.


## Cell 1 — Install dependencies

In [1]:
import subprocess, sys

packages = [
    "faster-whisper",
    "chatterbox-tts",
    "pydub",          # for .m4a → .wav conversion
    "sounddevice",
    "soundfile",
    "numpy",
    "scipy",
    "flask",
    "gradio>=4.16",
    "requests",
]

for p in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])

print("✅ All packages installed")
print("⚠️  Note: ffmpeg is required for .m4a conversion.")
print("   Install via: brew install ffmpeg   (Mac)")



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ All packages installed
⚠️  Note: ffmpeg is required for .m4a conversion.
   Install via: brew install ffmpeg   (Mac)



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Cell 2 — Imports & device setup

In [2]:
import os, time, threading, queue, json, re, uuid, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import soundfile as sf
import sounddevice as sd
from faster_whisper import WhisperModel
from chatterbox.tts import ChatterboxTTS
import torch
import requests
from flask import Flask, jsonify
import gradio as gr

warnings.filterwarnings("ignore")

# Auto-detect best device: MPS (Mac M4) → CUDA → CPU
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"🖥️  Device: {DEVICE}")


/Users/inkgabriel/Downloads/nursebot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️  Device: mps


## Cell 3 — Prepare your voice reference

Converts `.m4a` → `.wav` automatically if needed.

In [3]:
from pathlib import Path
import subprocess, shutil

VOICE_REF_PATH = "my_voice.wav"

# Find ffmpeg (Homebrew installs to /opt/homebrew/bin on M-series Macs)
FFMPEG = shutil.which("ffmpeg") or "/opt/homebrew/bin/ffmpeg"

if not Path(VOICE_REF_PATH).exists():
    if Path("my_voice.m4a").exists():
        print("🔄 Converting my_voice.m4a → my_voice.wav ...")
        result = subprocess.run([
            FFMPEG, "-y",
            "-i", "my_voice.m4a",
            "-ar", "22050",    # sample rate Chatterbox likes
            "-ac", "1",        # mono
            VOICE_REF_PATH
        ], capture_output=True, text=True)
        
        if result.returncode != 0:
            print("❌ ffmpeg error:")
            print(result.stderr)
        else:
            import soundfile as sf
            data, sr = sf.read(VOICE_REF_PATH)
            duration = len(data) / sr
            print(f"✅ Converted → {VOICE_REF_PATH} ({duration:.1f} sec @ {sr} Hz)")
    else:
        print("⚠️  No voice file found!")
        print("   Place my_voice.m4a next to this notebook and re-run this cell.")
else:
    import soundfile as sf
    data, sr = sf.read(VOICE_REF_PATH)
    duration = len(data) / sr
    print(f"✅ Voice reference ready: {VOICE_REF_PATH} ({duration:.1f} sec @ {sr} Hz)")
    if duration < 6:
        print("⚠️  Recording too short (<6 sec). 15–30 sec gives much better cloning.")

🔄 Converting my_voice.m4a → my_voice.wav ...
✅ Converted → my_voice.wav (6.2 sec @ 22050 Hz)


## Cell 4 — TTS Module (Chatterbox voice clone)

In [ ]:
class TTSModule:
    def __init__(self, voice_ref=VOICE_REF_PATH, device=DEVICE):
        print(f"🔊 Loading Chatterbox TTS on {device} ...")
        # Chatterbox uses 'cuda' or 'cpu' string; map mps → cpu for compatibility
        cb_device = device if device in ("cuda", "cpu") else "cpu"
        self.model = ChatterboxTTS.from_pretrained(device=cb_device)
        self.voice_ref = voice_ref
        self.sample_rate = self.model.sr
        self._lock = threading.Lock()
        print(f"✅ Chatterbox ready — voice ref: {voice_ref}")

    def speak(self, text, output_path=None, exaggeration=0.4, cfg_weight=0.5):
        """Synthesize text in the cloned voice and play it."""
        if not text.strip():
            return
        output_path = output_path or f"nurse_response_{uuid.uuid4().hex[:6]}.wav"
        with self._lock:
            wav = self.model.generate(
                text,
                audio_prompt_path=self.voice_ref,
                exaggeration=exaggeration,
                cfg_weight=cfg_weight,
            )
            audio = wav.squeeze().cpu().numpy()
            sf.write(output_path, audio, self.sample_rate)
        # Play audio
        data, sr = sf.read(output_path)
        sd.play(data, sr)
        sd.wait()
        return output_path

    def speak_async(self, text):
        """Speak without blocking the main thread."""
        t = threading.Thread(target=self.speak, args=(text,), daemon=True)
        t.start()


tts = TTSModule()
print("\n🎤 Test — nurse will speak in your voice:")
tts.speak("Hello. I am your nurse assistant. Voice clone is active.")


🔊 Loading Chatterbox TTS on mps ...


## Cell 5 — STT Module (Faster-Whisper)

In [ ]:
class STTModule:
    def __init__(self, model_size="base", device=DEVICE):
        wh_device = "cpu" if device == "mps" else device
        compute = "int8" if wh_device == "cpu" else "float16"
        print(f"🎙️  Loading Whisper '{model_size}' on {wh_device} ...")
        self.model = WhisperModel(model_size, device=wh_device, compute_type=compute)
        self.sample_rate = 16000
        print("✅ Whisper ready")

    def listen(self, duration=5, threshold=0.01):
        """Record from mic and transcribe."""
        print(f"🎤 Listening for {duration} sec...")
        audio = sd.rec(int(duration * self.sample_rate),
                       samplerate=self.sample_rate, channels=1, dtype="float32")
        sd.wait()
        audio = audio.flatten()
        if np.abs(audio).max() < threshold:
            return ""
        tmp = f"_listen_{uuid.uuid4().hex[:6]}.wav"
        sf.write(tmp, audio, self.sample_rate)
        segments, _ = self.model.transcribe(tmp, beam_size=5)
        text = " ".join(s.text for s in segments).strip()
        os.remove(tmp)
        print(f"📝 Heard: '{text}'")
        return text

    def transcribe_file(self, path):
        segments, _ = self.model.transcribe(path, beam_size=5)
        return " ".join(s.text for s in segments).strip()


stt = STTModule()


## Cell 6 — Task Manager

In [ ]:
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional

class Priority(Enum):
    EMERGENCY = 0
    HIGH = 1
    NORMAL = 2
    LOW = 3

class TaskStatus(Enum):
    QUEUED = "queued"
    RUNNING = "running"
    COMPLETED = "completed"
    INTERRUPTED = "interrupted"
    CANCELLED = "cancelled"

@dataclass
class Task:
    id: str
    description: str
    room: Optional[str]
    priority: Priority
    role: str          # "doctor" or "patient"
    status: TaskStatus = TaskStatus.QUEUED
    created_at: str = field(default_factory=lambda: datetime.now().strftime("%H:%M:%S"))
    completed_at: Optional[str] = None

class TaskManager:
    def __init__(self):
        self.tasks: list[Task] = []
        self._lock = threading.Lock()

    def add(self, description, room=None, priority=Priority.NORMAL, role="doctor") -> Task:
        task = Task(
            id=uuid.uuid4().hex[:6].upper(),
            description=description,
            room=room,
            priority=priority,
            role=role,
        )
        with self._lock:
            self.tasks.append(task)
            self.tasks.sort(key=lambda t: t.priority.value)
        return task

    def cancel(self, keyword) -> Optional[Task]:
        with self._lock:
            for t in self.tasks:
                if (keyword.lower() in t.description.lower() and
                        t.status in (TaskStatus.QUEUED, TaskStatus.RUNNING, TaskStatus.INTERRUPTED)):
                    t.status = TaskStatus.CANCELLED
                    return t
        return None

    def interrupt(self) -> Optional[Task]:
        with self._lock:
            for t in self.tasks:
                if t.status == TaskStatus.RUNNING:
                    t.status = TaskStatus.INTERRUPTED
                    return t
        return None

    def complete_current(self):
        with self._lock:
            for t in self.tasks:
                if t.status == TaskStatus.RUNNING:
                    t.status = TaskStatus.COMPLETED
                    t.completed_at = datetime.now().strftime("%H:%M:%S")
                    return t
        return None

    def next_task(self) -> Optional[Task]:
        with self._lock:
            for t in self.tasks:
                if t.status == TaskStatus.QUEUED:
                    t.status = TaskStatus.RUNNING
                    return t
        return None

    def status_summary(self) -> str:
        with self._lock:
            active = [t for t in self.tasks if t.status == TaskStatus.RUNNING]
            queued = [t for t in self.tasks if t.status == TaskStatus.QUEUED]
            done   = [t for t in self.tasks if t.status == TaskStatus.COMPLETED]
        lines = []
        if active:
            lines.append(f"Running: {active[0].description} (ID {active[0].id})")
        if queued:
            lines.append(f"Queued ({len(queued)}): " + ", ".join(t.description for t in queued[:3]))
        if done:
            lines.append(f"Completed today: {len(done)}")
        return ". ".join(lines) if lines else "No active tasks."

    def to_json(self):
        with self._lock:
            return [
                {"id": t.id, "description": t.description, "room": t.room,
                 "priority": t.priority.name, "status": t.status.value,
                 "role": t.role, "created_at": t.created_at}
                for t in self.tasks
            ]


tm = TaskManager()
print("✅ TaskManager ready")


## Cell 7 — Intent Parser

In [ ]:
ROOM_MAP = {
    "301": "Room 301", "302": "Room 302", "303": "Room 303",
    "304": "Room 304", "305": "Room 305",
    "icu": "ICU", "lab": "Laboratory", "medicine": "Medicine Cabinet",
    "medication": "Medicine Cabinet", "pharmacy": "Pharmacy", "nurses": "Nurses Station",
}

def extract_room(text):
    text_l = text.lower()
    for key, val in ROOM_MAP.items():
        if key in text_l:
            return val
    return None

def classify_priority(text):
    text_l = text.lower()
    if any(w in text_l for w in ["urgent", "emergency", "critical", "immediately", "chest pain", "not breathing"]):
        return Priority.EMERGENCY
    if any(w in text_l for w in ["high priority", "asap", "quickly"]):
        return Priority.HIGH
    if any(w in text_l for w in ["low priority", "when available", "no rush"]):
        return Priority.LOW
    return Priority.NORMAL

def ollama_intent(text, role):
    """Try Ollama for smarter intent; fall back to keyword parser."""
    try:
        r = requests.post("http://localhost:11434/api/generate", json={
            "model": "llama3.2",
            "prompt": (
                f"You are parsing a voice command from a {role} to a robot nurse. "
                f"Command: \"{text}\". "
                "Reply ONLY with JSON: {"action": "add_task|cancel_task|interrupt|status|unknown", "
                ""description": "task description", "room": "room or null", "
                ""priority": "EMERGENCY|HIGH|NORMAL|LOW"}"
            ),
            "stream": False,
        }, timeout=5)
        raw = r.json().get("response", "")
        match = re.search(r"\{.*?\}", raw, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return None

def parse_intent(text, role="doctor"):
    text_l = text.lower()
    ollama = ollama_intent(text, role)

    if ollama and ollama.get("action") not in (None, "unknown"):
        action = ollama["action"]
        desc = ollama.get("description", text)
        room = ollama.get("room") or extract_room(text)
        priority = Priority[ollama.get("priority", "NORMAL")]
    else:
        # Keyword fallback
        room = extract_room(text)
        priority = classify_priority(text)
        if any(w in text_l for w in ["cancel", "remove", "delete", "stop"]):
            action, desc = "cancel_task", text
        elif any(w in text_l for w in ["interrupt", "pause", "hold"]):
            action, desc = "interrupt", text
        elif any(w in text_l for w in ["status", "what tasks", "update", "report"]):
            action, desc = "status", text
        elif any(w in text_l for w in ["check", "administer", "medication", "blood", "temperature",
                                        "help", "pain", "draw", "vitals", "measure", "bring"]):
            action, desc = "add_task", text
        else:
            action, desc = "unknown", text

    return {"action": action, "description": desc, "room": room, "priority": priority}


print("✅ Intent parser ready (Ollama → keyword fallback)")


## Cell 8 — Robot state & Flask map server

In [ ]:
ROOMS = {
    "Medicine Cabinet": (120, 80), "Laboratory":       (300, 80),
    "ICU":              (480, 80), "Room 301":         (80,  220),
    "Room 302":         (200, 220), "Room 303":        (320, 220),
    "Room 304":         (80,  340), "Room 305":        (200, 340),
    "Nurses Station":   (400, 300), "Pharmacy":        (480, 340),
}

robot_state = {
    "position": list(ROOMS["Nurses Station"]),
    "target": None,
    "current_task": None,
    "status": "idle",
}

def move_robot(target_room):
    if target_room not in ROOMS:
        return
    robot_state["target"] = target_room
    robot_state["status"] = "moving"
    tx, ty = ROOMS[target_room]
    steps = 30
    sx, sy = robot_state["position"]
    for i in range(steps + 1):
        robot_state["position"] = [
            sx + (tx - sx) * i / steps,
            sy + (ty - sy) * i / steps,
        ]
        time.sleep(0.05)
    robot_state["status"] = "working"
    time.sleep(2)
    robot_state["status"] = "idle"
    robot_state["target"] = None

flask_app = Flask(__name__)

MAP_HTML = '''<!DOCTYPE html>
<html><head><title>Nurse Robot Map</title>
<style>
  body { background:#1a1a2e; font-family: Arial, sans-serif; display:flex; justify-content:center; align-items:center; height:100vh; margin:0; }
  canvas { border: 2px solid #16213e; border-radius:8px; }
</style></head>
<body>
<canvas id="c" width="580" height="440"></canvas>
<script>
const ROOMS = ''' + json.dumps(ROOMS) + ''';
const canvas = document.getElementById("c");
const ctx = canvas.getContext("2d");
let robot = {x:400, y:300};

function drawMap(state) {
  ctx.fillStyle = "#16213e";
  ctx.fillRect(0, 0, canvas.width, canvas.height);

  // Draw rooms
  for (const [name, [x, y]] of Object.entries(ROOMS)) {
    const isTarget = name === state.target;
    ctx.fillStyle = isTarget ? "#e94560" : "#0f3460";
    ctx.strokeStyle = "#533483";
    ctx.lineWidth = 2;
    ctx.beginPath();
    ctx.roundRect(x-50, y-25, 100, 50, 6);
    ctx.fill(); ctx.stroke();
    ctx.fillStyle = "#eee";
    ctx.font = "11px Arial";
    ctx.textAlign = "center";
    ctx.fillText(name, x, y+4);
  }

  // Draw robot
  ctx.beginPath();
  ctx.arc(state.position[0], state.position[1], 14, 0, Math.PI*2);
  ctx.fillStyle = state.status === "moving" ? "#f5a623" : state.status === "working" ? "#7ed321" : "#4a90e2";
  ctx.fill();
  ctx.strokeStyle = "#fff"; ctx.lineWidth = 2; ctx.stroke();
  ctx.fillStyle = "#fff"; ctx.font = "bold 12px Arial"; ctx.textAlign = "center";
  ctx.fillText("R", state.position[0], state.position[1]+4);

  // Status
  ctx.fillStyle = "#eee"; ctx.font = "13px Arial"; ctx.textAlign = "left";
  ctx.fillText("Status: " + state.status.toUpperCase(), 10, 430);
  if (state.current_task) ctx.fillText("Task: " + state.current_task, 10, 415);
}

async function poll() {
  try {
    const r = await fetch("/state");
    const s = await r.json();
    drawMap(s);
  } catch(e) {}
  setTimeout(poll, 300);
}
poll();
</script></body></html>'''

@flask_app.route("/")
def index():
    return MAP_HTML

@flask_app.route("/state")
def state():
    return jsonify({**robot_state, "tasks": tm.to_json()})

def run_flask():
    import logging
    log = logging.getLogger("werkzeug")
    log.setLevel(logging.ERROR)
    flask_app.run(port=7861, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(1)
print("🗺️  Map running at http://localhost:7861")


## Cell 9 — Nurse System (command processor)

In [ ]:
class NurseSystem:
    def __init__(self, task_manager, tts_module):
        self.tm = task_manager
        self.tts = tts_module

    def process(self, text, role="doctor"):
        if not text.strip():
            return ""
        intent = parse_intent(text, role)
        action = intent["action"]

        if action == "add_task":
            task = self.tm.add(
                description=intent["description"],
                room=intent["room"],
                priority=intent["priority"],
                role=role,
            )
            room_str = f" in {task.room}" if task.room else ""
            response = f"Understood. Task {task.id} added{room_str}: {task.description}."
            if task.priority == Priority.EMERGENCY:
                current = self.tm.interrupt()
                if current:
                    response += f" Interrupting current task to handle emergency."
                threading.Thread(
                    target=move_robot, args=(task.room,), daemon=True
                ).start()

        elif action == "cancel_task":
            keyword = re.sub(r"cancel|remove|delete|stop", "", text, flags=re.IGNORECASE).strip()
            cancelled = self.tm.cancel(keyword or text)
            response = (f"Task cancelled: {cancelled.description}." if cancelled
                        else "No matching task found to cancel.")

        elif action == "interrupt":
            interrupted = self.tm.interrupt()
            response = (f"Pausing current task: {interrupted.description}." if interrupted
                        else "No task is currently running.")

        elif action == "status":
            response = self.tm.status_summary()

        else:
            response = "I didn't understand that command. Please try again."

        print(f"[{role.upper()}] → {response}")
        self.tts.speak_async(response)

        # Auto-move robot for normal tasks
        if action == "add_task" and intent["room"] and intent["priority"] != Priority.EMERGENCY:
            threading.Thread(
                target=move_robot, args=(intent["room"],), daemon=True
            ).start()

        return response


nurse = NurseSystem(tm, tts)
print("✅ Nurse system ready")


## Cell 10 — Gradio Control Panel (launch here)

In [ ]:
def handle_text(text, role):
    if not text.strip():
        return "Please type a command.", task_table()
    response = nurse.process(text, role.lower())
    return response, task_table()

def handle_voice(audio, role):
    if audio is None:
        return "No audio received.", task_table()
    path, data = audio
    transcribed = stt.transcribe_file(path)
    if not transcribed:
        return "Could not understand audio.", task_table()
    response = nurse.process(transcribed, role.lower())
    return f"[Heard: {transcribed}]\n{response}", task_table()

def task_table():
    tasks = tm.to_json()
    if not tasks:
        return "No tasks yet."
    rows = ["| ID | Priority | Status | Room | Description |",
            "|---|---|---|---|---|"]
    for t in tasks[-10:]:
        rows.append(f"| {t['id']} | {t['priority']} | {t['status']} | {t['room'] or '—'} | {t['description'][:40]} |")
    return "\n".join(rows)

QUICK = [
    ("🩺 Check BP Room 302",    "Check blood pressure for patient in room 302"),
    ("💊 Medication Room 305",  "Administer evening medication in room 305"),
    ("🌡️ Temperature Room 304", "Measure temperature in room 304"),
    ("🚨 Emergency Room 301",   "Urgent! Patient in room 301 has chest pains"),
    ("📋 Task Status",          "What tasks are active?"),
    ("❌ Cancel Last Task",     "Cancel the current task"),
]

def quick_cmd(cmd_text, role):
    return handle_text(cmd_text, role)

with gr.Blocks(title="🏥 Nurse Robot", theme=gr.themes.Soft()) as ui:
    gr.Markdown("# 🏥 Robot Nurse — Voice Clone Control\n*Nurse speaks in your voice*")

    with gr.Row():
        role_radio = gr.Radio(["Doctor", "Patient"], value="Doctor", label="Your role")

    with gr.Tabs():
        with gr.Tab("💬 Text Command"):
            text_in = gr.Textbox(placeholder="e.g. Check blood pressure room 302", label="Command")
            text_btn = gr.Button("Send", variant="primary")
            text_out = gr.Textbox(label="Nurse Response", interactive=False)

        with gr.Tab("🎤 Voice Command"):
            audio_in = gr.Audio(sources=["microphone"], type="filepath", label="Speak your command")
            voice_btn = gr.Button("Process Voice", variant="primary")
            voice_out = gr.Textbox(label="Nurse Response", interactive=False)

    gr.Markdown("### ⚡ Quick Commands")
    with gr.Row():
        for label, cmd in QUICK[:3]:
            btn = gr.Button(label, size="sm")
            btn.click(fn=lambda c=cmd, r=role_radio: quick_cmd(c, r),
                      inputs=[], outputs=[text_out])
    with gr.Row():
        for label, cmd in QUICK[3:]:
            btn = gr.Button(label, size="sm")
            btn.click(fn=lambda c=cmd, r=role_radio: quick_cmd(c, r),
                      inputs=[], outputs=[text_out])

    task_display = gr.Markdown(label="Task Queue", value="No tasks yet.")
    refresh_btn = gr.Button("🔄 Refresh Tasks")

    text_btn.click(handle_text, inputs=[text_in, role_radio], outputs=[text_out, task_display])
    voice_btn.click(handle_voice, inputs=[audio_in, role_radio], outputs=[voice_out, task_display])
    refresh_btn.click(task_table, outputs=[task_display])

    gr.Markdown("---\n🗺️ **[Open Hospital Map](http://localhost:7861)** — watch the robot move in real time")

ui.launch(server_port=7860, share=False, inbrowser=True)
